# Geocoding, 날씨 API 활용

크롤링 없는 실습

pip install geopy

In [2]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Geocoding 클라이언트 초기화
# 반드시 user_agent를 설정해야 한다. (본인의 프로젝트명 등으로 설정)
# Nominatim 서비스를 사용하며, API Key는 필요 없다.
try:
    geolocator = Nominatim(user_agent="geocoding_analysis_tool")

    address = "부산광역시 부산진구 중앙대로 668"

    location = geolocator.geocode(address, timeout=10)

except Exception as e:
    print(f"API 요청 중 오류 발생: {e}")
    

latitude = location.latitude
longitude = location.longitude

In [6]:
latitude, longitude

(35.1524287, 129.0596192)

pip install openmeteo-requests retry-requests requests-cache

In [7]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
from datetime import datetime

# Open-Meteo 클라이언트 초기화
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# API 엔드포인트 URL
URL = "https://api.open-meteo.com/v1/forecast"

# 날씨 API 요청 (위경도 기반)

# 요청 매개변수 (Parameters) 설정
params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": ["temperature_2m", "relative_humidity_2m", "weather_code", "wind_speed_10m"],
    "timezone": "Asia/Seoul",
    "forecast_days": 1
}

try:
    # API 호출을 실행한다.
    responses = openmeteo.weather_api(URL, params=params)
    response = responses[0]
    
    # 데이터 추출 및 구조화 (Current Weather)
    current = response.Current()

    current_data = {
        "시간": datetime.fromtimestamp(current.Time()).strftime('%Y-%m-%d %H:%M:%S'),
        "온도_2m": current.Variables(0).Value(),
        "상대_습도_2m": current.Variables(1).Value(),
        "날씨_코드": current.Variables(2).Value(),
        "풍속_10m": current.Variables(3).Value()
    }

except Exception as e:
    print(f"API 요청 중 오류 발생: {e}")

In [8]:
current_data

{'시간': '2025-12-18 16:15:00',
 '온도_2m': 10.300000190734863,
 '상대_습도_2m': 46.0,
 '날씨_코드': 0.0,
 '풍속_10m': 4.072935104370117}

pip install python-dotenv

In [14]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

pip install google-genai

In [15]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words in Korean",
)

print(response.text)

**데이터를 학습하여 판단합니다.**

(De-i-teo-reul hak-seup-ha-yeo pan-dan-ham-ni-da.)

*Meaning:* It learns data and makes judgments/decisions.


In [16]:
prompt = f"""다음은 현재 날씨 정보 검색 결과입니다. 
이 결과를 분석하여 날씨 상황을 요약하고 설명해주세요.

**날씨 데이터:**
{current_data}
"""

print(prompt)

다음은 현재 날씨 정보 검색 결과입니다. 
이 결과를 분석하여 날씨 상황을 요약하고 설명해주세요.

**날씨 데이터:**
{'시간': '2025-12-18 16:15:00', '온도_2m': 10.300000190734863, '상대_습도_2m': 46.0, '날씨_코드': 0.0, '풍속_10m': 4.072935104370117}



In [17]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

제공된 날씨 데이터를 분석한 결과는 다음과 같습니다.

**날씨 상황 요약 및 설명:**

*   **관측 시간:** 2025년 12월 18일 오후 4시 15분
*   **기온:** 10.3°C
*   **상대 습도:** 46%
*   **날씨 상태:** 맑음 (날씨 코드 0.0은 일반적으로 맑은 하늘을 의미합니다.)
*   **풍속:** 4.07m/s

**종합적인 날씨 설명:**

현재 2025년 12월 18일 늦은 오후 4시 15분 기준으로 **맑고 온화한 날씨**를 보이고 있습니다.

기온은 **10.3°C**로, 12월 중순의 오후 시간대임을 감안할 때 비교적 포근한 편입니다. 영하로 떨어지지 않아 심하게 춥지는 않을 것으로 보입니다.

상대 습도는 **46%**로 보통 수준이며, 쾌적하게 느껴질 수 있는 범위입니다. 하늘은 **맑게 개어 있을 것**으로 예상되어 활동하기 좋은 날씨입니다.

풍속은 **4.07m/s**로, 약간 강한 바람이 부는 정도입니다. 이 바람 때문에 체감 온도는 실제 기온보다 약간 낮게 느껴질 수 있지만, 전반적으로 맑고 비교적 따뜻하여 겨울철임에도 불구하고 상쾌한 느낌을 줄 수 있는 날씨로 판단됩니다.


In [18]:
prompt = f"""다음은 현재 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**날씨 데이터:**
```json
{current_data}
```
"""

print(prompt)

다음은 현재 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**날씨 데이터:**
```json
{'시간': '2025-12-18 16:15:00', '온도_2m': 10.300000190734863, '상대_습도_2m': 46.0, '날씨_코드': 0.0, '풍속_10m': 4.072935104370117}
```



In [19]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

현재 날씨 데이터를 분석한 결과, 우산이 **필요하지 않습니다.**

**이유:**

*   `날씨_코드`가 **0.0**으로 되어 있습니다. 이는 일반적으로 '맑음' 또는 '구름 없음' 상태를 나타내며, 비나 눈이 오고 있지 않음을 의미합니다.

따라서 지금 외출하신다면 우산을 챙길 필요는 없습니다.


In [20]:
prompt = f"""다음은 현재 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**날씨 데이터:**
```json
{current_data}
```
"""

print(prompt)

다음은 현재 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**날씨 데이터:**
```json
{'시간': '2025-12-18 16:15:00', '온도_2m': 10.300000190734863, '상대_습도_2m': 46.0, '날씨_코드': 0.0, '풍속_10m': 4.072935104370117}
```



In [21]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

현재 날씨 데이터를 분석하여 지금 외출 시 추천하는 드레스 코디를 알려드리겠습니다.

**날씨 요약:**

*   **시간:** 2025년 12월 18일 16시 15분 (늦은 오후, 겨울철)
*   **기온:** 약 10.3°C
*   **습도:** 46.0% (보통)
*   **날씨:** 맑음 (날씨 코드 0.0)
*   **풍속:** 약 4.1 m/s (바람이 다소 부는 편)

**분석:**
12월 늦은 오후 10.3°C는 쌀쌀하며, 풍속 4.1m/s의 바람까지 불어 체감 온도는 더 낮을 것입니다. 해가 지기 시작하면 기온이 빠르게 떨어질 수 있으므로, 보온에 신경 쓴 옷차림이 필요합니다.

---

**지금 외출 시 추천 드레스 코디:**

바람이 불기 때문에 **외풍을 막아줄 수 있는 아우터**와 **겹겹이 입는 것**이 좋습니다.

1.  **아우터 (가장 중요):**
    *   **두꺼운 코트** (울 코트 등) 또는 **패딩 점퍼** (롱패딩, 숏패딩 모두 좋음). 바람막이 기능이 있는 아우터라면 더욱 좋습니다.
    *   **경량 패딩 단독은 추울 수 있습니다.**

2.  **상의:**
    *   **니트 스웨터** 또는 **기모 맨투맨/후드티**와 같이 따뜻한 긴팔 상의를 입으세요.
    *   혹은 긴팔 티셔츠 위에 가디건이나 조끼를 레이어드하는 것도 좋습니다.

3.  **하의:**
    *   **두꺼운 긴 바지** (청바지, 면바지, 슬랙스 등)를 추천합니다.
    *   찬 바람을 막아줄 수 있는 소재가 좋습니다. (기모 안감이면 더 좋습니다.)
    *   만약 치마를 입는다면, 반드시 **두꺼운 스타킹이나 기모 레깅스**를 착용하고 긴 부츠를 신는 것이 좋습니다.

4.  **액세서리 (보온 효과 UP):**
    *   **목도리:** 바람으로부터 목을 보호하고 체온 유지에 큰 도움이 됩니다.
    *   **모자:** 특히 귀마개 역할을 하는 모자라면 더욱 좋습니다.
    *   **장갑:** 손이 시릴

In [28]:
prompt = f"""다음은 현재 날씨 데이터입니다. 
이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요. 
결과는 현재 날씨에 어울리는 스타일로 HTML, CSS를 사용한 인포그래픽을 작성해주세요. HTML 문서 외 설명은 작성하지 마세요.
이모지, 아이콘을 사용하고, 이미지는 사용하지 마세요.

**날씨 데이터:**
```json
{current_data}
```
"""

print(prompt)

다음은 현재 날씨 데이터입니다. 
이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요. 
결과는 현재 날씨에 어울리는 스타일로 HTML, CSS를 사용한 인포그래픽을 작성해주세요. HTML 문서 외 설명은 작성하지 마세요.
이모지, 아이콘을 사용하고, 이미지는 사용하지 마세요.

**날씨 데이터:**
```json
{'시간': '2025-12-18 16:15:00', '온도_2m': 10.300000190734863, '상대_습도_2m': 46.0, '날씨_코드': 0.0, '풍속_10m': 4.072935104370117}
```



In [29]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

```html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>오늘의 외출 코디 추천</title>
    <link href="https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@300;400;700&display=swap" rel="stylesheet">
    <style>
        :root {
            --primary-color: #3498db; /* Blue */
            --secondary-color: #2c3e50; /* Darker blue/grey */
            --accent-color: #e67e22; /* Orange */
            --text-color: #333;
            --bg-light: #ecf0f1; /* Light grey */
            --bg-white: #ffffff;
            --border-color: #bdc3c7; /* Light grey border */
        }

        body {
            font-family: 'Noto Sans KR', sans-serif;
            background-color: var(--bg-light);
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            margin: 0;
            color: var(--text-color);
            line-h

In [30]:
result = response.text
result = result.replace("```html","").replace("```","")
print(result)


<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>오늘의 외출 코디 추천</title>
    <link href="https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@300;400;700&display=swap" rel="stylesheet">
    <style>
        :root {
            --primary-color: #3498db; /* Blue */
            --secondary-color: #2c3e50; /* Darker blue/grey */
            --accent-color: #e67e22; /* Orange */
            --text-color: #333;
            --bg-light: #ecf0f1; /* Light grey */
            --bg-white: #ffffff;
            --border-color: #bdc3c7; /* Light grey border */
        }

        body {
            font-family: 'Noto Sans KR', sans-serif;
            background-color: var(--bg-light);
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            margin: 0;
            color: var(--text-color);
            line-height: 

In [31]:
current_data

{'시간': '2025-12-18 16:15:00',
 '온도_2m': 10.300000190734863,
 '상대_습도_2m': 46.0,
 '날씨_코드': 0.0,
 '풍속_10m': 4.072935104370117}

In [32]:
current_data['시간'].split()[0].replace("-", "")

'20251218'

In [33]:
# html 인포그래픽 생성
formatted_time = current_data['시간'].split()[0].replace("-", "")

file = open(f"result_{formatted_time}.html", "w", encoding="utf8")

file.write(str(result))
file.close()